# Treine seu próprio mini-LLM

Neste notebook você vai treinar um modelo de linguagem do zero e depois ajustar um modelo pronto do Hugging Face.

**Parte 1, do zero:** um mini-GPT nasce sem saber nada e aprende a escrever. Vamos mudar a base de dados, o tamanho do modelo
e a temperatura, treinar com poucos dados e olhar a atenção por dentro.

**Parte 2, ajuste fino:** um modelo pré-treinado aprende as regras do Banco Aurora, uma empresa fictícia, para responder clientes.

Antes de começar: **Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4**.

In [ ]:
import gc, json, os, sys, time, warnings
warnings.filterwarnings("ignore")

IN_COLAB = "google.colab" in sys.modules
RAPIDO = os.environ.get("MINI_RAPIDO") == "1"   # teste automatizado, com menos passos
REPO = "https://github.com/pgcferreira-stack/aula-mini-llm-fgv"

if IN_COLAB:
    !pip install -q peft
    if not os.path.exists("aula-mini-llm-fgv"):
        !git clone -q --depth 1 {REPO}
    os.chdir("aula-mini-llm-fgv")

import matplotlib.pyplot as plt
import pandas as pd
import torch
from tokenizers import ByteLevelBPETokenizer
from transformers import GPT2Config, GPT2LMHeadModel

DEVICE = os.environ.get("MINI_DEVICE") or ("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
torch.manual_seed(295)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
print("Device:", DEVICE)

# Parte 1 — Um LLM treinado do zero

## 1.1 As bases de dados

Temos duas bases de texto: um romance de Machado de Assis, em domínio público, e conversas de atendimento bancário.

In [ ]:
machado = open("dados/machado.txt", encoding="utf-8").read()
atendimento = open("dados/atendimento.txt", encoding="utf-8").read()

print(f"Machado de Assis: {len(machado):,} caracteres")
print(machado[3000:3250])
print(f"\nAtendimento: {len(atendimento):,} caracteres")
print(atendimento[:250])

## 1.2 De texto a tokens

O modelo não lê letras: lê tokens. Aqui treinamos um tokenizador que aprende, a partir do próprio texto, quais pedaços de palavra são frequentes.

In [ ]:
def criar_tokenizador(texto, vocabulario=1500):
    os.makedirs("/tmp/mini", exist_ok=True)
    with open("/tmp/mini/texto.txt", "w", encoding="utf-8") as f:
        f.write(texto)
    tok = ByteLevelBPETokenizer()
    tok.train(["/tmp/mini/texto.txt"], vocab_size=vocabulario, min_frequency=2, show_progress=False)
    return tok

def preparar(texto, vocabulario=1500):
    """Devolve o tokenizador e os tokens separados em treino (90%) e validação (10%)."""
    tok = criar_tokenizador(texto, vocabulario)
    ids = torch.tensor(tok.encode(texto).ids, dtype=torch.long)
    corte = int(len(ids) * 0.9)
    return tok, ids[:corte], ids[corte:]

tok_machado, treino_machado, val_machado = preparar(machado)
exemplo = "Abri uma conta no banco e a tarifa veio errada."
print("Tokens:", [tok_machado.decode([i]) for i in tok_machado.encode(exemplo).ids])
print(f"A base tem {len(treino_machado) + len(val_machado):,} tokens; o vocabulário tem {tok_machado.get_vocab_size():,}.")

## 1.3 O modelo

Usamos a arquitetura do GPT-2, do Hugging Face, mas com pesos aleatórios: nada foi baixado. Três números definem o tamanho do modelo:
camadas, cabeças de atenção e dimensão dos embeddings.

In [ ]:
BLOCO = 96   # quantos tokens o modelo enxerga de cada vez

def criar_modelo(vocabulario, camadas=4, cabecas=4, dimensao=192):
    config = GPT2Config(vocab_size=vocabulario, n_positions=BLOCO, n_layer=camadas, n_head=cabecas, n_embd=dimensao,
                        attn_implementation="eager")
    modelo = GPT2LMHeadModel(config).to(DEVICE)
    print(f"Modelo com {sum(p.numel() for p in modelo.parameters()):,} parâmetros")
    return modelo

modelo = criar_modelo(tok_machado.get_vocab_size())

## 1.4 Treinar e gerar

O treino é sempre o mesmo ciclo: o modelo tenta prever o próximo token, medimos o erro e ajustamos um pouco os pesos.
Medimos o erro também em um trecho que o modelo não usa para aprender, a **validação**.

In [ ]:
def lote(ids, tamanho=32):
    inicio = torch.randint(0, len(ids) - BLOCO - 1, (tamanho,))
    return torch.stack([ids[i:i + BLOCO] for i in inicio]).to(DEVICE)

@torch.no_grad()
def erro(modelo, ids, vezes=20):
    modelo.eval()
    valores = [modelo(input_ids=(x := lote(ids)), labels=x).loss.item() for _ in range(vezes)]
    modelo.train()
    return sum(valores) / len(valores)

def treinar(modelo, treino, validacao, passos=3000, taxa=1e-3, medir_a_cada=250, ao_medir=None):
    otimizador = torch.optim.AdamW(modelo.parameters(), lr=taxa)
    historia, t0 = [], time.time()
    modelo.train()
    for passo in range(passos + 1):
        if passo % medir_a_cada == 0:
            historia.append({"passo": passo, "treino": erro(modelo, treino), "validacao": erro(modelo, validacao)})
            if ao_medir:
                ao_medir(passo)
        if passo == passos:
            break
        x = lote(treino)
        perda = modelo(input_ids=x, labels=x).loss     # 1. prever e medir o erro
        otimizador.zero_grad()
        perda.backward()                               # 2. atribuir a culpa a cada peso
        otimizador.step()                              # 3. ajustar os pesos
    print(f"{passos} passos em {time.time() - t0:.0f}s | erro de treino {historia[-1]['treino']:.2f} | erro de validação {historia[-1]['validacao']:.2f}")
    return pd.DataFrame(historia)

@torch.no_grad()
def gerar(modelo, tok, inicio, tokens=60, temperatura=0.8, top_k=40):
    modelo.eval()
    x = torch.tensor([tok.encode(inicio).ids], device=DEVICE)
    saida = modelo.generate(x, max_new_tokens=tokens, do_sample=True, temperature=temperatura, top_k=top_k, pad_token_id=0)
    modelo.train()
    return tok.decode(saida[0].tolist())

## Experimento 1: o modelo nasce sem saber nada

**Antes de rodar:** o que você espera ler no passo 0? E depois de 3.000 passos?

In [ ]:
PASSOS = 300 if RAPIDO else 3000

def mostrar(passo):
    if passo in (0, 500, PASSOS // 3, PASSOS):
        print(f"\n--- Passo {passo} ---\n{gerar(modelo, tok_machado, 'Quando eu', 40)}")

hist_machado = treinar(modelo, treino_machado, val_machado, PASSOS, ao_medir=mostrar)

In [ ]:
plt.plot(hist_machado.passo, hist_machado.treino, label="treino")
plt.plot(hist_machado.passo, hist_machado.validacao, label="validação")
plt.xlabel("passos de treino"); plt.ylabel("erro"); plt.title("Erro de treino e de validação"); plt.legend(); plt.show()

## Experimento 2: mesmo modelo, outra base

Mudamos só os dados. A arquitetura é idêntica.

**Antes de rodar:** o que o mesmo início de frase vai gerar em cada modelo?

In [ ]:
tok_atend, treino_atend, val_atend = preparar(atendimento)
modelo_atend = criar_modelo(tok_atend.get_vocab_size())
hist_atend = treinar(modelo_atend, treino_atend, val_atend, PASSOS)

for inicio in ["Cliente: Qual o valor", "Cliente: Como bloqueio"]:
    print(f"\nInício: {inicio!r}")
    print("  Treinado com Machado:", gerar(modelo, tok_machado, inicio, 40).replace("\n", " "))
    print("  Treinado com atendimento:", gerar(modelo_atend, tok_atend, inicio, 40).replace("\n", " "))

## Experimento 3: tamanho do modelo

Vamos comparar três tamanhos, com a mesma base e o mesmo número de passos.

**Antes de rodar:** maior é sempre melhor? Observe o erro de validação, os parâmetros e o tempo.

In [ ]:
TAMANHOS = {"Pequeno": (2, 2, 64), "Médio": (4, 4, 192), "Grande": (6, 6, 300)}
PASSOS_TAMANHO = 200 if RAPIDO else 1500
linhas = []
for nome, (camadas, cabecas, dimensao) in TAMANHOS.items():
    print(f"\n{nome}:")
    teste = criar_modelo(tok_machado.get_vocab_size(), camadas, cabecas, dimensao)
    t0 = time.time()
    h = treinar(teste, treino_machado, val_machado, PASSOS_TAMANHO, medir_a_cada=PASSOS_TAMANHO)
    linhas.append({"modelo": nome, "camadas": camadas, "cabeças": cabecas, "dimensão": dimensao,
                   "parâmetros": sum(p.numel() for p in teste.parameters()), "tempo (s)": round(time.time() - t0),
                   "erro de validação": round(h.validacao.iloc[-1], 2)})
    print(gerar(teste, tok_machado, "Quando eu", 30).replace("\n", " "))
    del teste
pd.DataFrame(linhas)

## Experimento 4: e se tivermos poucos dados?

Treinamos o mesmo modelo com apenas 3% do livro e acompanhamos o erro de treino e o de validação.

**Antes de rodar:** o que deve acontecer com cada curva?

In [ ]:
pequeno = treino_machado[:int(len(treino_machado) * 0.03)]
print(f"Base completa: {len(treino_machado):,} tokens | base reduzida: {len(pequeno):,} tokens")
modelo_pouco = criar_modelo(tok_machado.get_vocab_size())
hist_pouco = treinar(modelo_pouco, pequeno, val_machado, 200 if RAPIDO else 2000, medir_a_cada=100)

fig, eixos = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
eixos[0].plot(hist_machado.passo, hist_machado.treino, label="treino"); eixos[0].plot(hist_machado.passo, hist_machado.validacao, label="validação")
eixos[0].set_title("Base completa"); eixos[0].legend(); eixos[0].set_xlabel("passos")
eixos[1].plot(hist_pouco.passo, hist_pouco.treino, label="treino"); eixos[1].plot(hist_pouco.passo, hist_pouco.validacao, label="validação")
eixos[1].set_title("3% da base"); eixos[1].legend(); eixos[1].set_xlabel("passos")
plt.show()

## Experimento 5: temperatura

A temperatura controla quanto risco o modelo assume ao escolher a próxima palavra.

**Antes de rodar:** quando você usaria uma temperatura baixa? E alta?

In [ ]:
for temperatura in [0.2, 0.8, 1.5]:
    print(f"\nTemperatura {temperatura}:")
    for _ in range(2):
        print(" ", gerar(modelo, tok_machado, "Quando eu", 30, temperatura, top_k=0).replace("\n", " "))

## Experimento 6: a atenção por dentro

Pedimos ao modelo os pesos de atenção de uma frase. Cada linha mostra para onde um token olha.

**Antes de rodar:** por que metade do gráfico deve ficar vazia?

In [ ]:
@torch.no_grad()
def mapa_de_atencao(modelo, tok, frase):
    modelo.eval()
    ids = tok.encode(frase).ids
    saida = modelo(input_ids=torch.tensor([ids], device=DEVICE), output_attentions=True)
    rotulos = [tok.decode([i]).replace("\n", "↵") for i in ids]
    n = len(saida.attentions)
    fig, eixos = plt.subplots(1, 2, figsize=(14, 6))
    for eixo, camada, nome in [(eixos[0], 0, "Primeira camada"), (eixos[1], n - 1, "Última camada")]:
        pesos = saida.attentions[camada][0].mean(dim=0).cpu().numpy()
        eixo.imshow(pesos, cmap="Blues", vmin=0)
        eixo.set_xticks(range(len(rotulos)), rotulos, rotation=75, fontsize=9)
        eixo.set_yticks(range(len(rotulos)), rotulos, fontsize=9)
        eixo.set_title(nome); eixo.grid(False)
    plt.tight_layout(); plt.show()

mapa_de_atencao(modelo_atend, tok_atend, "Cliente: Abri uma conta no banco e a tarifa veio errada")

# Parte 2 — Ajuste fino de um modelo pré-treinado

Agora o problema é real: responder clientes do Banco Aurora, uma empresa fictícia, com **apenas 150 exemplos**.
Vamos comparar dois caminhos: ajustar um modelo pronto e treinar outro do zero.

## 2.1 A base de perguntas e respostas

In [ ]:
for nome in ["modelo", "modelo_atend", "modelo_pouco"]:
    globals().pop(nome, None)
gc.collect(); torch.cuda.empty_cache() if DEVICE == "cuda" else None

faq = [json.loads(linha) for linha in open("dados/faq_banco_aurora.jsonl", encoding="utf-8")]
print(len(faq), "pares de pergunta e resposta. Exemplo:")
print(faq[0])

TESTES = [("A tarifa veio errada. Podem estornar?", ["50", "5 dias"]),
          ("Meu cartão sumiu. Como eu bloqueio?", ["aplicativo"]),
          ("Qual é o valor mensal para manter minha conta?", ["19,90"]),
          ("Se eu pagar a fatura depois do vencimento, qual é a multa?", ["2%"]),
          ("Quero encerrar minha conta. Vou pagar alguma coisa?", ["gratuito"])]

## 2.2 Um modelo pronto, antes do ajuste

Baixamos o Qwen2.5-0.5B-Instruct, um LLM aberto de cerca de 500 milhões de parâmetros. Ele sabe conversar, mas não conhece as regras do banco.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

NOME_MODELO = "Qwen/Qwen2.5-0.5B-Instruct"
INSTRUCAO = "Você é o assistente do Banco Aurora. Responda em português, em uma ou duas frases."
tok_qwen = AutoTokenizer.from_pretrained(NOME_MODELO)
qwen = AutoModelForCausalLM.from_pretrained(NOME_MODELO, dtype=torch.float32).to(DEVICE)

def conversa(pergunta, resposta=None):
    mensagens = [{"role": "system", "content": INSTRUCAO}, {"role": "user", "content": pergunta}]
    if resposta is not None:
        mensagens.append({"role": "assistant", "content": resposta})
    return tok_qwen.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=resposta is None)

@torch.no_grad()
def responder(pergunta):
    qwen.eval()
    x = tok_qwen(conversa(pergunta), return_tensors="pt").to(DEVICE)
    saida = qwen.generate(**x, max_new_tokens=60, do_sample=False)
    return tok_qwen.decode(saida[0][x.input_ids.shape[1]:], skip_special_tokens=True).strip()

antes = {p: responder(p) for p, _ in TESTES}
for p, r in antes.items():
    print(f"P: {p}\nR: {r}\n")

## 2.3 Ajuste fino com LoRA

O LoRA congela o modelo e treina apenas pequenas matrizes adicionais. Por isso, o ajuste é rápido e barato.

In [ ]:
from peft import LoraConfig, get_peft_model

qwen = get_peft_model(qwen, LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, task_type="CAUSAL_LM",
                                       target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
qwen.print_trainable_parameters()

def exemplo_de_treino(par):
    inicio = tok_qwen(conversa(par["pergunta"])).input_ids
    completo = tok_qwen(conversa(par["pergunta"], par["resposta"])).input_ids
    rotulos = [-100] * len(inicio) + completo[len(inicio):]    # o erro só é medido na resposta
    return torch.tensor(completo), torch.tensor(rotulos)

EPOCAS, LOTE = (1, 4) if RAPIDO else (6, 4)
exemplos = [exemplo_de_treino(par) for par in faq]
otimizador = torch.optim.AdamW([p for p in qwen.parameters() if p.requires_grad], lr=1e-3)
qwen.train(); t0 = time.time()
for epoca in range(EPOCAS):
    ordem, total = torch.randperm(len(exemplos)).tolist(), 0
    for i in range(0, len(ordem), LOTE):
        grupo = [exemplos[j] for j in ordem[i:i + LOTE]]
        x = torch.nn.utils.rnn.pad_sequence([g[0] for g in grupo], batch_first=True, padding_value=tok_qwen.pad_token_id).to(DEVICE)
        y = torch.nn.utils.rnn.pad_sequence([g[1] for g in grupo], batch_first=True, padding_value=-100).to(DEVICE)
        perda = qwen(input_ids=x, attention_mask=(x != tok_qwen.pad_token_id).long(), labels=y).loss
        otimizador.zero_grad(); perda.backward(); otimizador.step()
        total += perda.item() * len(grupo)
    print(f"Época {epoca + 1}: erro {total / len(exemplos):.2f} ({time.time() - t0:.0f}s)")

In [ ]:
depois = {p: responder(p) for p, _ in TESTES}
for p, r in depois.items():
    print(f"P: {p}\nAntes:  {antes[p]}\nDepois: {r}\n")

## 2.4 O mesmo problema, treinado do zero

Agora treinamos um mini-GPT apenas com as mesmas 150 perguntas e respostas.

**Antes de rodar:** quantos casos cada modelo vai acertar?

In [ ]:
texto_faq = "\n\n".join(f"Cliente: {par['pergunta']}\nAtendente: {par['resposta']}" for par in faq)
tok_faq = criar_tokenizador(texto_faq, 500)
ids_faq = torch.tensor(tok_faq.encode(texto_faq).ids, dtype=torch.long)
print(f"Base de {len(texto_faq):,} caracteres e {len(ids_faq):,} tokens")
modelo_faq = criar_modelo(tok_faq.get_vocab_size())
_ = treinar(modelo_faq, ids_faq, ids_faq, 300 if RAPIDO else 1500, medir_a_cada=500)

def responder_mini(pergunta):
    torch.manual_seed(295)
    texto = gerar(modelo_faq, tok_faq, f"Cliente: {pergunta}\nAtendente:", 45, temperatura=0.5, top_k=20)
    return texto.split("Atendente:", 1)[1].split("Cliente")[0].strip()

mini = {p: responder_mini(p) for p, _ in TESTES}

## 2.5 Comparação final

Uma resposta conta como acerto quando contém a informação-chave do caso. É uma medida simples, mas objetiva.

In [ ]:
def acertou(resposta, chaves):
    return all(c.lower() in resposta.lower() for c in chaves)

tabela = pd.DataFrame([{"pergunta": p, "pronto, sem ajuste": antes[p], "pronto, com ajuste fino": depois[p], "mini-GPT do zero": mini[p]} for p, _ in TESTES])
pd.set_option("display.max_colwidth", None)
display(tabela)

for coluna, respostas in [("Pronto, sem ajuste", antes), ("Pronto, com ajuste fino", depois), ("Mini-GPT do zero", mini)]:
    print(f"{coluna}: {sum(acertou(respostas[p], chaves) for p, chaves in TESTES)} de {len(TESTES)} acertos")

## Para discutir

1. O que mudou mais o resultado: a arquitetura, os dados ou a temperatura?
2. Por que o erro de validação sobe com poucos dados?
3. Por que o modelo pronto precisa de bem menos dados para aprender as regras?
4. Esse sistema pode ir para produção? O que você mudaria?